In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from arch import arch_model
from scipy.optimize import minimize

In [2]:
#plot 한글 적용
plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

In [3]:
def getAssetsStock(tickers, startDate, endDate):
    
    dic = {}
    file_path = "../Data-collection/dailyStock"
    
    for ticker in tickers:
        ticker_df = pd.read_csv(f"{file_path}/{ticker}.csv", encoding="utf-8")
        ticker_df['Date'] = pd.to_datetime(ticker_df['Date'])
        ticker_df = ticker_df.set_index('Date')
        ticker_df = ticker_df.loc[startDate:endDate]
        
        dic[ticker] = 100 * ticker_df['adj_close'].pct_change().dropna()
    
    returns_df = pd.DataFrame(dic)
    
    return returns_df.dropna()

In [13]:
def dcc_recursion(params, Z, Q_bar):

    a, b = params
    T, N = Z.shape
    Q_t = Q_bar.copy()

    Q_list = []
    R_list = []

    for t in range(T):

        if t > 0:

            z_prev = Z[t - 1].reshape(-1, 1)

            Q_t = (
                (1 - a - b) * Q_bar
                + a * (z_prev @ z_prev.T)
                + b * Q_t
            )

        diag = np.sqrt(np.diag(Q_t))

        R_t = (
            Q_t /
            np.outer(diag, diag)
        )

        Q_list.append(Q_t.copy())
        R_list.append(R_t.copy())

    return np.array(Q_list), np.array(R_list)

In [14]:
def dcc_likelihood(params, Z, Q_bar):

    a, b = params

    if a < 0 or b < 0 or a + b >= 1:
        return 1e10

    _, R_list = dcc_recursion(
        params,
        Z,
        Q_bar
    )

    likelihood = 0.0

    for t in range(len(Z)):

        R_t = R_list[t]
        z_t = Z[t]

        sign, logdet = np.linalg.slogdet(R_t)

        if sign <= 0:
            return 1e10

        try:
            inv_R = np.linalg.inv(R_t)
        except np.linalg.LinAlgError:
            return 1e10

        likelihood += (
            logdet
            + z_t @ inv_R @ z_t
        )

    return likelihood

### 데이터 불러오기(2015~2018년도)

In [4]:
RISKY_ASSETS = ["AAPL", "GOOG", "MSFT"]
N = len(RISKY_ASSETS)
START_DATE = "2015-01-01"
END_DATE = "2018-12-31"

In [5]:
returns = getAssetsStock(RISKY_ASSETS, START_DATE, END_DATE)

### 종목별 GARCH(1,1) 추정

In [6]:
models = []
std_resids = []
cond_vol = []

In [7]:
for asset in returns.columns:

    model = arch_model(returns[asset], mean='Constant', vol='GARCH', p=1, o=0, q=1, dist='normal').fit(disp='off')

    cond_vol.append(model.conditional_volatility) # 조건부 변동성
    std_resids.append(model.resid / model.conditional_volatility) # 표준화 잔차
    models.append(model)

In [8]:
cond_vol_df = pd.DataFrame(dict(zip(returns.columns, cond_vol)),index=returns.index)
std_resids_df = pd.DataFrame(dict(zip(returns.columns, std_resids)),index=returns.index)

### 표준화 잔차 준비

In [12]:
Z_df = std_resids_df.dropna()
Z = Z_df.values
T, N = Z.shape

# Q-bar 계산
Q_bar = (Z.T @ Z) / T

### DCC 파라미터 추정

In [17]:
result = minimize(
    dcc_likelihood,
    x0=np.array([0.05, 0.90]),
    args=(Z, Q_bar),
    method='SLSQP',
    bounds=[
        (1e-6, 0.999),
        (1e-6, 0.999)
    ],
    constraints={
        'type': 'ineq',
        'fun': lambda x: 0.999 - x[0] - x[1]
    }
)


a, b = result.x

print('DCC alpha :', a)
print('DCC beta  :', b)
print('alpha+beta:', a + b)
print('converged :', result.success)

DCC alpha : 0.0598965058425944
DCC beta  : 0.7922988213382451
alpha+beta: 0.8521953271808396
converged : True
